In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from tqdm import tqdm
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
%load_ext autoreload
%autoreload 2
from drGAT import drGAT
from drGAT.load_data import load_data
from drGAT.sampler import BalancedSampler
from drGAT.myutils import get_all_edges_and_labels
from drGAT.metrics import compute_metrics_stats

In [3]:
method = "GATv2"
data = 'nci'

In [15]:
def suggest_hyperparams(trial, S_d, S_c, S_g):
    hidden1 = trial.suggest_int("hidden1", 256, 1024)
    hidden2 = trial.suggest_int("hidden2", 64, min(512, hidden1))
    hidden3 = trial.suggest_int("hidden3", 32, min(256, hidden2))

    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_float("dropout1", 0.1, 0.5, step=0.05),
        "dropout2": trial.suggest_float("dropout2", 0.1, 0.5, step=0.05),
        "hidden1": hidden1,
        "hidden2": hidden2,
        "hidden3": hidden3,
        "epochs": 3,
         # trial.suggest_int("epochs", 100, 10000, step=100),
        "heads": trial.suggest_int("heads", 2, 8),
        "activation": trial.suggest_categorical("activation", ["relu", "gelu"]),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical("scheduler", [None, "Cosine"]),
        "norm_type": trial.suggest_categorical("norm_type", ["GraphNorm", "BatchNorm", "LayerNorm"]),
        "gnn_layer": method,
    }

    # Scheduler-specific parameter
    if params["scheduler"] == "Cosine":
        min_epoch_div = max(1, params["epochs"] // 5)
        max_epoch_div = max(min_epoch_div + 1, params["epochs"] // 2)
        params["T_max"] = trial.suggest_int("T_max", low=min_epoch_div, high=max_epoch_div)

        if params["T_max"] <= 0:
            raise optuna.TrialPruned(f"Invalid T_max: {params['T_max']}")

    return params

In [16]:
def handle_optuna_errors(e, trial):
    msg = str(e)
    if "CUDA out of memory" in msg:
        print(f"Pruned trial {trial.number}: CUDA OOM")
        with torch.cuda.device("cuda"):
            torch.cuda.empty_cache()
        gc.collect()
        raise optuna.TrialPruned(f"OOM at trial {trial.number}")
    elif "Input contains NaN" in msg:
        print(f"Pruned trial {trial.number}: Input contains NaN")
        raise optuna.TrialPruned(f"NaN input at trial {trial.number}")
    elif isinstance(e, ZeroDivisionError):
        print(f"Pruned trial {trial.number}: ZeroDivisionError in CosineAnnealingLR")
        raise optuna.TrialPruned("ZeroDivisionError in CosineAnnealingLR")
    else:
        print(f"Unexpected error in trial {trial.number}: {msg}")
        raise e

In [17]:
def objective(trial):
    try:
        # === Data Load ===
        is_zero_pad = trial.suggest_categorical("is_zero_pad", [True, False])
        drugAct, null_mask, S_d, S_c, S_g, drug_feature, gene_norm_gene, gene_norm_cell, A_cg, A_dg = load_data(
            data, is_zero_pad=is_zero_pad
        )

        # === Suggest Hyperparameters ===
        params = suggest_hyperparams(trial, S_d, S_c, S_g)

        # === Prepare K-Fold Sampling ===
        all_edges, all_labels = get_all_edges_and_labels(drugAct, null_mask)
        kf = KFold(n_splits=5, shuffle=True, random_state=42)

        true_datas = pd.DataFrame()
        predict_datas = pd.DataFrame()

        for train_idx, test_idx in tqdm(kf.split(all_edges)):
            sampler = BalancedSampler(
                drugAct, all_edges, all_labels, train_idx, test_idx,
                null_mask, S_d, S_c, S_g, A_cg, A_dg
            )

            (
                _,
                _,
                _,
                predict_data,
                true_data,
                _,
                _,
                _,
                _,
            ) = drGAT.train(sampler, params=params, device=device, verbose=False)

            true_datas = pd.concat([true_datas, pd.DataFrame(true_data).T], ignore_index=True)
            predict_datas = pd.concat([predict_datas, pd.DataFrame(predict_data).T], ignore_index=True)

        # === Compute Final Metrics ===
        metrics_result = compute_metrics_stats(
            trial=trial,
            true=true_datas,
            pred=predict_datas,
            target_metrics=["AUROC", "AUPR", "F1", "ACC"],
        )
        return tuple(metrics_result["target_values"])

    except Exception as e:
        handle_optuna_errors(e, trial)

In [18]:
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.NSGAIISampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage=f"sqlite:///{method}_{data}_small.sqlite3",
    study_name=method,
    load_if_exists=True,
)
study.optimize(objective, n_trials=1000)

[I 2025-05-05 17:29:33,631] Using an existing study with name 'GATv2' instead of creating a new one.


load nci
Done!


0it [00:00, ?it/s]

Using device: cpu


1it [02:42, 162.02s/it]

Best model found at epoch 3
[0.8092759  0.8255897  0.8221891  ... 0.8152325  0.8198933  0.81167597]
Using device: cpu


1it [04:13, 253.85s/it]
[W 2025-05-05 17:33:49,935] Trial 8 failed with parameters: {'is_zero_pad': False, 'hidden1': 260, 'hidden2': 98, 'hidden3': 84, 'dropout1': 0.25, 'dropout2': 0.4, 'heads': 5, 'activation': 'relu', 'optimizer': 'Adam', 'lr': 0.004483430801049571, 'weight_decay': 1.010598145637589e-06, 'scheduler': 'Cosine', 'norm_type': 'GraphNorm', 'T_max': 2} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/y3/ssnk1ytd3m5bjmrchh2lt74srg76p8/T/ipykernel_72394/3837836806.py", line 35, in objective
    ) = drGAT.train(sampler, params=params, device=device, verbose=False)
  File "/Users/inouey2/code/drGAT/drGAT/drGAT.py", line 317, in train
    train_attention = train_one_epoch(
  File "/Users/inouey2/code/drGAT/drGAT/drGAT.py", line 503, in train_one_epoch
    los

KeyboardInterrupt: 

,number,values_0,values_1,values_2,values_3,datetime_start,datetime_complete,duration,params_T_max,params_activation,...,params_hidden2,params_hidden3,params_is_zero_pad,params_lr,params_norm_type,params_optimizer,params_scheduler,params_weight_decay,system_attrs_nsga2:generation,state
0,0,None,None,None,None,2025-05-05 17:09:49.507823,2025-05-05 17:10:11.957972,0 days 00:00:22.450149,NaN,relu,...,365.0,107.0,NaN,0.000276,NaN,Adam,None,0.005108,0.0,FAIL
1,1,None,None,None,None,2025-05-05 17:14:48.808729,2025-05-05 17:14:48.817151,0 days 00:00:00.008422,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FAIL
2,2,None,None,None,None,2025-05-05 17:17:47.737715,NaT,NaT,NaN,relu,...,207.0,51.0,False,0.000011,NaN,Adam,None,0.000507,0.0,RUNNING
3,3,None,None,None,None,2025-05-05 17:24:14.230977,2025-05-05 17:24:20.461716,0 days 00:00:06.230739,1.0,relu,...,88.0,42.0,True,0.009370,GraphNorm,AdamW,Cosine,0.000441,0.0,FAIL
4,4,None,None,None,None,2025-05-05 17:24:42.103189,2025-05-05 17:24:48.032263,0 days 00:00:05.929074,NaN,relu,...,71.0,44.0,True,0.006110,BatchNorm,Adam,None,0.000121,0.0,FAIL
5,5,None,None,None,None,2025-05-05 17:25:26.644683,2025-05-05 17:26:19.213358,0 days 00:00:52.568675,2.0,gelu,...,145.0,145.0,False,0.000180,LayerNorm,AdamW,Cosine,0.000015,0.0,FAIL
6,6,None,None,None,None,2025-05-05 17:26:21.365712,2025-05-05 17:27:02.781120,0 days 00:00:41.415408,NaN,gelu,...,260.0,254.0,True,0.005427,GraphNorm,Adam,None,0.000026,0.0,FAIL
7,7,None,None,None,None,2025-05-05 17:29:13.733079,2025-05-05 17:29:16.127434,0 days 00:00:02.394355,NaN,NaN,...,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,0.0,FAIL
8,8,None,None,None,None,2025-05-05 17:29:33.642163,2025-05-05 17:33:49.931841,0 days 00:04:16.289678,2.0,relu,...,98.0,84.0,False,0.004483,GraphNorm,Adam,Cosine,0.000001,0.0,FAIL
